In [ ]:
!pip install transformers datasets scikit-learn torch pandas numpy -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ai_pile_balanced.csv')

print(df.shape)
print(df['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

df = df[['text', 'label']].dropna()
df['text'] = df['text'].astype(str)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df['label']
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

train_encodings = tokenize(train_df['text'])
val_encodings = tokenize(val_df['text'])
test_encodings = tokenize(test_df['text'])

print("Tokenization completed!")

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TextDataset(train_encodings, train_df['label'])
val_dataset = TextDataset(val_encodings, val_df['label'])
test_dataset = TextDataset(test_encodings, test_df['label'])

print("Dataset created!")

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

print("DistilBERT model loaded!")

In [ ]:

from transformers import Trainer, TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=['Human', 'AI']))

print("\nConfusion Matrix:")
print(confusion_matrix(labels, preds))

In [ ]:
model_path = '/content/drive/MyDrive/distilbert_ai_pile_model'

model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print("Model saved to:", model_path)

In [ ]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

model_path = '/content/drive/MyDrive/distilbert_ai_pile_model'

tokenizer = DistilBertTokenizer.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(model_path)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

print("Model loaded on:", device)

def predict_text(text):
    if len(text.split()) < 10:
        print("⚠️ Text too short. Please enter longer text.")
        return

    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
        confidence = probs[0][prediction].item()

    label = "👤 Human Written" if prediction == 0 else "🤖 AI Generated"

    print("Prediction:", label)
    print(f"Confidence: {confidence * 100:.2f}%")

while True:
    text = input("\nEnter text to check or type quit: ")

    if text.lower() == 'quit':
        print("Exiting...")
        break

    predict_text(text)